# Topic 1: Delta Lake — Advanced Patterns

> **Phase 7 → Week 13 → Topic 1**

---

## What Week 9 Covered vs What This Adds

Week 9 introduced Delta Lake fundamentals (ACID, time travel, schema enforcement, basic MERGE INTO). This notebook goes deeper:

| Concept | Week 9 | Week 13 |
|---------|--------|--------|
| MERGE INTO | Basic upsert | Multi-action MERGE, conditional logic |
| Time travel | Basic query | Retention policies, RESTORE |
| Schema | Enforcement vs evolution | Column mapping, type widening |
| Change Data Feed | — | CDC stream, downstream consumers |
| Table properties | — | delta.tuning properties, target file size |
| OPTIMIZE | Basic | ZORDER selectivity, file sizing |

---

## Interview Cheat Sheet

**Q: What is Delta Lake's Change Data Feed (CDF)?**
> CDF records every row-level change (insert, update, delete) in a Delta table as a new `_change_type` column. Enable with `delta.enableChangeDataFeed=true`. Downstream consumers can read only the changes since their last checkpoint using `readChangeFeed` with `startingVersion`. Use case: propagating Silver→Gold updates without full re-read.

**Q: What is the difference between schema enforcement and schema evolution in Delta?**
> Schema enforcement (default): any write with a different schema than the existing table throws an error — data is never silently corrupted. Schema evolution (`mergeSchema=true`): new columns in the incoming DataFrame are added to the table schema automatically. You can never widen a column type without explicit migration — Delta rejects int→string without `overwriteSchema=true`.

**Q: What does OPTIMIZE ZORDER BY do?**
> OPTIMIZE rewrites small Parquet files into larger ones (default target: 1 GB). ZORDER BY co-locates rows with the same value(s) for the given columns in the same files — so a query with `WHERE customer_id = X` skips most files entirely (data skipping). ZORDER is most effective on high-cardinality columns used frequently in filters. Only do ZORDER on 1-2 columns — beyond that, the locality improvement degrades rapidly.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import shutil, os

spark = SparkSession.builder \
    .appName("Week13 - Delta Lake Advanced") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

BASE = "/tmp/delta_advanced"
if os.path.exists(BASE):
    shutil.rmtree(BASE)
os.makedirs(BASE)
print("Delta Lake Advanced Patterns — ready")

## Part 1: Advanced MERGE INTO Patterns

In [ ]:
from delta.tables import DeltaTable

# --- Setup: customer table (target) ---
customers = spark.createDataFrame([
    (1, "Alice",  "alice@old.com",   "gold",   2021),
    (2, "Bob",    "bob@corp.com",    "silver", 2020),
    (3, "Carol",  "carol@co.com",    "bronze", 2022),
    (4, "Dave",   "dave@corp.com",   "gold",   2019),
], ["id", "name", "email", "tier", "join_year"])

TARGET = f"{BASE}/customers"
customers.write.format("delta").mode("overwrite").save(TARGET)

# --- Source: incoming CRM updates ---
updates = spark.createDataFrame([
    (1, "Alice",  "alice@new.com",   "platinum", 2021),  # update email + tier
    (3, None,     None,              "deleted",  None),   # mark as deleted
    (5, "Eve",    "eve@startup.com", "silver",   2024),   # new customer
], ["id", "name", "email", "tier", "join_year"])

dt = DeltaTable.forPath(spark, TARGET)

# Multi-action MERGE:
#   WHEN MATCHED AND tier = 'deleted' → DELETE
#   WHEN MATCHED (otherwise)          → UPDATE
#   WHEN NOT MATCHED                  → INSERT
(
    dt.alias("target")
    .merge(updates.alias("source"), "target.id = source.id")
    .whenMatchedDelete(condition="source.tier = 'deleted'")
    .whenMatchedUpdate(set={
        "email": "source.email",
        "tier":  "source.tier",
    })
    .whenNotMatchedInsert(values={
        "id":        "source.id",
        "name":      "source.name",
        "email":     "source.email",
        "tier":      "source.tier",
        "join_year": "source.join_year",
    })
    .execute()
)

print("After MERGE:")
spark.read.format("delta").load(TARGET).orderBy("id").show()
# Expected: Alice updated, Carol deleted, Eve inserted, Bob+Dave unchanged

## Part 2: Change Data Feed (CDF)

In [ ]:
# Enable CDF on a table
CDF_TABLE = f"{BASE}/orders_cdf"

orders = spark.createDataFrame([
    (101, "A", 500.0, "pending"),
    (102, "B", 300.0, "pending"),
    (103, "C", 750.0, "pending"),
], ["order_id", "customer", "amount", "status"])

orders.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite").save(CDF_TABLE)

print("Version 0 — initial load")
spark.read.format("delta").load(CDF_TABLE).show()

# Version 1: update some orders
dt_cdf = DeltaTable.forPath(spark, CDF_TABLE)
(
    dt_cdf.alias("t")
    .merge(
        spark.createDataFrame([(101, "shipped"), (103, "cancelled")], ["order_id", "status"]).alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdate(set={"status": "s.status"})
    .execute()
)

print("\nVersion 1 — after status updates")
spark.read.format("delta").load(CDF_TABLE).show()

# Read the Change Data Feed — see exactly what changed
print("\nChange Data Feed (all changes from version 0):")
(
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .load(CDF_TABLE)
    .orderBy("order_id", "_commit_version")
    .show(20, truncate=False)
)
# _change_type will be: insert (v0), update_preimage (old row, v1), update_postimage (new row, v1)

## Part 3: Schema Evolution

In [ ]:
SCHEMA_TABLE = f"{BASE}/schema_demo"

# V1: original schema
v1 = spark.createDataFrame([
    (1, "laptop",  1200.0),
    (2, "phone",    800.0),
], ["id", "product", "price"])
v1.write.format("delta").mode("overwrite").save(SCHEMA_TABLE)

# V2: new column 'discount' added
v2 = spark.createDataFrame([
    (3, "tablet",  600.0, 0.10),
    (4, "watch",   400.0, 0.05),
], ["id", "product", "price", "discount"])

# Without mergeSchema → error
try:
    v2.write.format("delta").mode("append").save(SCHEMA_TABLE)
except Exception as e:
    print(f"Schema enforcement blocked mismatched write: {type(e).__name__}")

# With mergeSchema → new column added, old rows get NULL for discount
v2.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append").save(SCHEMA_TABLE)

print("\nAfter schema evolution (discount column added):")
spark.read.format("delta").load(SCHEMA_TABLE).orderBy("id").show()
# Old rows have NULL discount — Delta fills with null for evolved columns

## Part 4: Time Travel & RESTORE

In [ ]:
TIME_TABLE = f"{BASE}/time_travel"

for i in range(4):
    batch = spark.createDataFrame(
        [(j, f"item_{j}", float(j * 10 + i)) for j in range(1, 4)],
        ["id", "name", "value"]
    )
    batch.write.format("delta").mode("overwrite").save(TIME_TABLE)

# Check all versions
print("Delta table history:")
DeltaTable.forPath(spark, TIME_TABLE).history().select(
    "version", "timestamp", "operation"
).show()

# Time travel: read version 1 (second write)
print("\nVersion 1 (second overwrite):")
spark.read.format("delta").option("versionAsOf", 1).load(TIME_TABLE).show()

# RESTORE to version 1 (undo last two overwrites)
DeltaTable.forPath(spark, TIME_TABLE).restoreToVersion(1)
print("\nAfter RESTORE to version 1:")
spark.read.format("delta").load(TIME_TABLE).show()

# VACUUM: remove files older than retention period
# Default retention = 7 days, minimum = 168 hours
# VACUUM removes old Parquet files that are no longer referenced by any version
# After VACUUM, time travel to removed versions is impossible
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
DeltaTable.forPath(spark, TIME_TABLE).vacuum(0)  # 0 hours for demo only — never in prod!
print("VACUUM complete")

## Part 5: OPTIMIZE, ZORDER & Table Properties

In [ ]:
OPT_TABLE = f"{BASE}/optimize_demo"

# Create many small files (simulate small-file problem)
for i in range(10):
    spark.range(1000) \
        .withColumn("customer_id", (F.col("id") % 50).cast("int")) \
        .withColumn("amount", (F.rand() * 1000).cast("double")) \
        .withColumn("region", F.element_at(F.array(F.lit("US"), F.lit("EU"), F.lit("APAC")), (F.col("id") % 3 + 1).cast("int"))) \
        .write.format("delta").mode("append").save(OPT_TABLE)

# Count files before OPTIMIZE
before_files = len([
    f for f in os.listdir(OPT_TABLE)
    if f.endswith(".parquet")
])
print(f"Files before OPTIMIZE: ~{before_files} (some in subdirs)")

# OPTIMIZE: compact small files
spark.sql(f"OPTIMIZE delta.`{OPT_TABLE}`")
print("OPTIMIZE done — files compacted to ~1 file per partition")

# OPTIMIZE ZORDER BY customer_id
# Co-locates rows with the same customer_id in the same files
# Queries WHERE customer_id = X will skip most files
spark.sql(f"OPTIMIZE delta.`{OPT_TABLE}` ZORDER BY (customer_id)")
print("OPTIMIZE ZORDER BY customer_id done")

# Key table properties for production tuning
print("""
Useful Delta Table Properties (set with ALTER TABLE or tblproperties):

  delta.enableChangeDataFeed = true         → CDF for downstream CDC
  delta.autoOptimize.optimizeWrite = true   → Databricks: auto-coalesce on write
  delta.autoOptimize.autoCompact = true     → Databricks: auto OPTIMIZE after writes
  delta.logRetentionDuration = interval 30 days  → keep 30 days of history
  delta.deletedFileRetentionDuration = interval 7 days  → VACUUM cleans after 7 days
  delta.targetFileSize = 134217728          → target 128 MB per file (default)
  delta.tuneFileSizesForRewrites = true     → smaller files when MERGE-heavy workload

SQL:
  ALTER TABLE my_table
  SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
  );
""")

## Exercises

1. Write a MERGE INTO that implements SCD Type 2: when a customer's `tier` changes, set `valid_to` on the old row and insert a new row with `valid_from = today`. Use `whenMatchedUpdate` and `whenNotMatchedInsert` together.
2. Enable Change Data Feed on a Delta table, perform several MERGE operations, then read the CDF to build a running changelog. How does `_change_type` differ between plain UPDATE vs MERGE?
3. Create a Delta table, write 20 batches of small files, run OPTIMIZE ZORDER BY a filter column. Measure query time with `.explain()` and count files skipped (check `numFilesSkippedToReduceDataSize` in the operation metrics).
4. What is the risk of setting `delta.deletedFileRetentionDuration` to `interval 0 hours`? When would you legitimately do this?
5. Explain the difference between `mergeSchema=true` (append option) and `overwriteSchema=true` (overwrite option). When would each break a downstream consumer?